# Gerar imagens de anotações vindas do GroundThuth

Autor:  Viviane da Rosa Sommer

Atualizado: 07/04/2025

## Objetivo: A partir da imagem original encontrar a imagem de anotação de mesmo ID.


## Importações Necessárias

In [ ]:
import glob
import json
import os
import shutil

## Declaração de Constantes e Modelos

In [ ]:
ANNOTATION_PATH = "Bucket-Output-Versions/imagens-lote-4-5/rotulacao-lotes-4-5-chain-3/annotations/consolidated-annotation/consolidation-request/iteration-1/*.json"
IMAGE_PATH = "Bucket-Output-Versions/imagens-lote-4-5/rotulacao-lotes-4-5-chain-3/annotations/consolidated-annotation/output/**"
INPUT_PATH = "Bucket-Input-Versions/imagens-lote-4-5/"
OUTPUT_PATH = "Datasets/Lote-4-5-Incompleto/"

## Funções Necessárias

In [ ]:
def copy_files(original_image_path: str, file_image: str) -> None:
    """
    Copy files from input path to output path.

    Args:
        original_image_path (str): Original image path.
        file_image (str): File image path.

    Returns:
        None
    """
    try:
        original_image_path = original_image_path.replace("'", "_").replace("’", "_")
        destination_image = os.path.join(OUTPUT_PATH, original_image_path)
        destination_label = os.path.join(OUTPUT_PATH, "label_" + original_image_path)
        shutil.copy(os.path.join(INPUT_PATH, original_image_path), destination_image)
        shutil.copy(file_image, destination_label)
        print(f"Files copied:\n Image: {destination_image}\n Label: {destination_label}")
    except Exception as e:
        print(f"Error copying files: {e}")

def process_annotations() -> None:
    """
    Process annotations and copy files.

    Returns:
        None
    """
    try:
        for filename in glob.glob(ANNOTATION_PATH):
            with open(filename, 'r') as file:
                data = json.load(file)
                dataset_object_id = data[0]["datasetObjectId"]
                original_image_path = data[0]["dataObject"]["s3Uri"].split("/")[-1]
                image_found = False
                for file_image in glob.glob(IMAGE_PATH):
                    image_id = os.path.basename(file_image).split("_")[0]
                    if image_id == dataset_object_id:
                        copy_files(original_image_path, file_image)
                        image_found = True
                        break
                if not image_found:
                    print(f"Corresponding image not found for ID: {dataset_object_id}")
    except Exception as e:
        print(f"Error processing annotations: {e}")

## Gerar imagem de anotação

In [ ]:
process_annotations()

In [ ]:
!jupyter nbconvert --to html Generate_label_images.ipynb